In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts


/opt/python/lib/python3.7/site-packages/anndata/_core/anndata.py:21: FutureWarning: pandas.core.index is deprecated and will be removed in a future version.  The public classes are available in the top-level namespace.
  from pandas.core.index import RangeIndex


In [2]:
atlas_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/atlas_data.h5ad")

In [3]:
query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/harmonized_ipf_data.h5ad")

### set labels to use for integration:

check which cells have only level 1 annotations:

In [4]:
lev1_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 1, :].copy()
print("Number of cells with only level 1 annotation:", lev1_obs.shape[0])

Number of cells with only level 1 annotation: 62705


In [5]:
# count number of cells per ann_level_1 group that have no further annotations
lev1_annotations = lev1_obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).rename(columns={"ann_level_1":"n_cells_w_only_l1_ann"})
# count total cells in each ann_level_1 group
lev1_annotations['total'] = query_data.obs.groupby("ann_level_1").agg({"ann_level_1":"count"}).loc[lev1_annotations.index,"ann_level_1"]

In [6]:
lev1_annotations

,n_cells_w_only_l1_ann,total
ann_level_1,,
Endothelial,5008,41915
Epithelial,3997,86479
Glial,0,36
Immune,963,427772
Stroma,52737,79885


We will let all "Stroma" keep their labels, and will set all other cells as "unlabeled" (since they only make up a small part of their group)

In [66]:
lev1_labeled = query_data.obs.index[(query_data.obs.ann_level_1 == "Stroma") & (query_data.obs.ann_highest_res==1)]

In [67]:
lev1_unlabeled = query_data.obs.index[[max_res == 1 and lev1label != "Stroma" for max_res, lev1label in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_1)]]

In [68]:
len(lev1_labeled)

52737

In [69]:
len(lev1_unlabeled)

9968

In [70]:
lev1_annotations.n_cells_w_only_l1_ann.sum() == len(lev1_labeled) + len(lev1_unlabeled)

True

Now check which cells have only up to level 2 annotations:

In [71]:
lev2_obs = query_data.obs.loc[query_data.obs.ann_highest_res == 2, :].copy()
print("Number of cells with only level 1/2 annotation:", lev2_obs.shape[0])

Number of cells with only level 1/2 annotation: 18919


In [72]:
# count number of cells per ann_level_2 group that have no further annotations
lev2_annotations = lev2_obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).rename(columns={"ann_level_2":"n_cells_w_only_l2_ann"})
# count total cells in each ann_level_2 group
lev2_annotations['total'] = query_data.obs.groupby("ann_level_2").agg({"ann_level_2":"count"}).loc[lev2_annotations.index,"ann_level_2"]

In [73]:
lev2_annotations

,n_cells_w_only_l2_ann,total
ann_level_2,,
1_Endothelial,0,5008
1_Epithelial,0,3997
1_Immune,0,963
1_Stroma,0,52737
Airway epithelium,0,53211
Alveolar epithelium,0,27023
Blood vessels,7280,30985
Chondrocytes,42,42
Fibroblast lineage,2936,22378


We will set all cells with level 2 annotations "Chondrocytes", "Fibroblast lineage", "Schwann cells", "Lymphatic EC", "Mesothelium", "Smooth muscle" to their level 2 annotations.

In [74]:
lev2_labels_to_keep = ["Chondrocytes", "Fibroblast lineage", "Schwann cells", "Lymphatic EC", "Mesothelium", "Smooth muscle"]
lev2_labeled = query_data.obs.index[[lev2_lab in lev2_labels_to_keep for lev2_lab in query_data.obs.ann_level_2]]

In [75]:
len(lev2_labeled) == lev2_annotations.loc[lev2_labels_to_keep,"total"].sum()

True

In [76]:
lev2_unlabeled = query_data.obs.index[[max_res == 2 and lev2_lab not in lev2_labels_to_keep for max_res, lev2_lab in zip(query_data.obs.ann_highest_res, query_data.obs.ann_level_2)]]

In [77]:
len(lev2_unlabeled) == lev2_annotations.loc[~lev2_annotations.index.isin(lev2_labels_to_keep),"n_cells_w_only_l2_ann"].sum()

True

now generate scANVI labels:

In [78]:
query_data.obs['scanvi_label'] = query_data.obs.ann_level_3.tolist()
query_data.obs.loc[lev1_labeled,'scanvi_label'] = query_data.obs.loc[lev1_labeled, "ann_level_1"]
query_data.obs.loc[lev1_unlabeled,'scanvi_label'] = "unlabeled"
query_data.obs.loc[lev2_labeled,'scanvi_label'] = query_data.obs.loc[lev2_labeled, "ann_level_2"]
query_data.obs.loc[lev2_unlabeled,'scanvi_label'] = "unlabeled"

In [79]:
[x for x in query_data.obs['scanvi_label'].unique() if x not in atlas_data.obs["scanvi_label"].unique()]

['Erythrocytes', 'Schwann cells', 'Chondrocytes', 'Stroma', 'Erythroblast']

save dataset 

In [83]:
query_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled.h5ad")

In [3]:
query_data = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled.h5ad")

### Normalize counts

In [4]:
#Save raw counts
query_data.layers["counts"] = query_data.X.copy()

In [8]:
query_data.obs.dataset.unique()

[Meyer_2021_5prime, Meyer_2021_3prime, Kaminski_2020, Banovich_Kropski_2020_disease, Misharin_Budinger_2018_disease, Sheppard_2020]
Categories (6, object): [Meyer_2021_5prime, Meyer_2021_3prime, Kaminski_2020, Banovich_Kropski_2020_disease, Misharin_Budinger_2018_disease, Sheppard_2020]

In [12]:
meyer_data = query_data[(query_data.obs.dataset=="Meyer_2021_5prime") | (query_data.obs.dataset=="Meyer_2021_5prime"),:]
adams = query_data[(query_data.obs.dataset=="Kaminski_2020"),:]
haberman = query_data[(query_data.obs.dataset=="Banovich_Kropski_2020_disease"),:]
reyfman = query_data[(query_data.obs.dataset=="Misharin_Budinger_2018_disease"),:]
tsukui = query_data[(query_data.obs.dataset=="Sheppard_2020"),:]


In [11]:
# normalize 
adams_norm = scib_excerpts.SCRAN_normalize(adams, log_transform=False)

In [13]:
# store
adams_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/adams_norm.h5ad")

In [14]:
# normalize 
haberman_norm = scib_excerpts.SCRAN_normalize(haberman, log_transform=False)

In [15]:
# store
haberman_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/haberman_norm.h5ad")

In [16]:
# normalize 
reyfman_norm = scib_excerpts.SCRAN_normalize(reyfman, log_transform=False)

In [17]:
# store
reyfman_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/reyfman_norm.h5ad")

In [18]:
# normalize 
tsukui_norm = scib_excerpts.SCRAN_normalize(tsukui, log_transform=False)

In [19]:
# store
tsukui_norm.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/Backup/tsukui_norm.h5ad")

In [20]:
final_tmp1 = adams_norm.concatenate(haberman_norm)
final_tmp2 = final_tmp1.concatenate(reyfman_norm)
final = final_tmp2.concatenate(tsukui_norm)

In [21]:
# store
final.write("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled_normalized.h5ad")

... storing 'ann_level_2' as categorical
... storing 'ann_level_2_clean' as categorical
... storing 'ann_level_3' as categorical
... storing 'ann_level_3_clean' as categorical
... storing 'ann_level_4' as categorical
... storing 'ann_level_4_clean' as categorical
... storing 'ann_level_5' as categorical
... storing 'ann_level_5_clean' as categorical
... storing 'condition' as categorical
... storing 'dataset' as categorical
... storing 'original_celltype_ann' as categorical
... storing 'sample' as categorical
... storing 'sample_type' as categorical
... storing 'scanvi_label' as categorical
... storing 'single_cell_platform' as categorical
... storing 'study' as categorical
... storing 'subject_ID' as categorical


In [2]:
final = sc.read("/storage/groups/ml01/workspace/jan.domke/Data/Models/IPF/IPF_labeled_normalized.h5ad")